# DMET 1001 — Image Processing Assignment 1
**German University in Cairo** | Dr. Mohamed Karam Gabr | **Due: 14 April 2026**

| Name | ID |
|---|---|
| Habiba | ??? |
| Fareeda | ??? |
| Nour | ??? |
| Haya | ??? |

---
## Cell 1 — Imports

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import pywt

print('All imports successful')

---
## Cell 2 — Unified Parameters
These values are identical across all 4 domain experiments. Do not change them inside a domain section.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
# We use 5 out of CIFAR-10's 10 classes — chosen for visual diversity
SELECTED_CLASSES = [0, 1, 2, 8, 9]        # airplane, automobile, bird, ship, truck
CLASS_NAMES      = ['airplane', 'automobile', 'bird', 'ship', 'truck']
NUM_CLASSES      = len(SELECTED_CLASSES)   # 5
DATA_DIR         = 'data'

# ── Split ─────────────────────────────────────────────────────────────────────
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15

# ── Training (must be identical across all 4 experiments) ─────────────────────
BATCH_SIZE    = 64
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
SEED          = 42

# ── Input sizes ───────────────────────────────────────────────────────────────
CIFAR_SIZE       = 32    # native CIFAR-10 image size
MODEL_INPUT_SIZE = 224   # MobileNet expects 224x224

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Classes     : {CLASS_NAMES}')
print(f'Num classes : {NUM_CLASSES}')
print(f'Device      : {device}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epochs      : {NUM_EPOCHS}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Optimizer   : Adam')
print(f'Split       : {int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/{int(TEST_RATIO*100)}')
print(f'Seed        : {SEED}')

---
## Cell 3 — Dataset Loader (shared)

In [ ]:
def get_dataloaders(transform_fn):
    """
    Downloads CIFAR-10, filters to SELECTED_CLASSES only, and returns
    train / val / test DataLoaders.
    Labels are remapped to 0–4 so the model always sees 0-indexed classes.
    The split is fixed by SEED so all 4 domains see the exact same images.
    """
    # Download CIFAR-10
    train_data = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=ToTensor())
    test_data  = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=ToTensor())

    # Map original class index → new 0-based index  e.g. {0:0, 1:1, 2:2, 8:3, 9:4}
    label_map = {orig: new for new, orig in enumerate(SELECTED_CLASSES)}

    def filter_and_remap(dataset):
        """Keep only images belonging to SELECTED_CLASSES and remap their labels."""
        indices, new_labels = [], []
        for i, lbl in enumerate(dataset.targets):
            if lbl in label_map:
                indices.append(i)
                new_labels.append(label_map[lbl])
        return indices, new_labels

    train_idx, train_labels = filter_and_remap(train_data)
    test_idx,  test_labels  = filter_and_remap(test_data)

    # 70/15 stratified split on the filtered training set
    rng    = np.random.default_rng(SEED)
    labels = np.array(train_labels)
    split_train_idx, split_val_idx = [], []

    for cls in range(NUM_CLASSES):
        idx = np.where(labels == cls)[0]
        rng.shuffle(idx)
        n = int(len(idx) * TRAIN_RATIO / (TRAIN_RATIO + VAL_RATIO))
        split_train_idx.extend(idx[:n].tolist())
        split_val_idx.extend(idx[n:].tolist())

    class _DS(Dataset):
        def __init__(self, base, indices, labels, fn):
            self.base, self.indices, self.labels, self.fn = base, indices, labels, fn
        def __len__(self): return len(self.indices)
        def __getitem__(self, i):
            img, _ = self.base[self.indices[i]]   # ignore original label
            return self.fn(img), self.labels[i]   # use remapped label

    train_labels = np.array(train_labels)
    train_ds = _DS(train_data, [train_idx[i] for i in split_train_idx], train_labels[split_train_idx], transform_fn)
    val_ds   = _DS(train_data, [train_idx[i] for i in split_val_idx],   train_labels[split_val_idx],   transform_fn)
    test_ds  = _DS(test_data,  test_idx, np.array(test_labels), transform_fn)

    print(f'train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}')

    kw = dict(batch_size=BATCH_SIZE, num_workers=2)
    return (DataLoader(train_ds, shuffle=True,  **kw),
            DataLoader(val_ds,   shuffle=False, **kw),
            DataLoader(test_ds,  shuffle=False, **kw))

print('get_dataloaders ready')

---
## Cell 4 — Model (shared)

In [ ]:
def get_model(in_channels=3):
    """
    Returns MobileNetV2 pretrained on ImageNet, adapted for CIFAR-10.
    in_channels: number of channels output by the domain transform.
      - Spatial / Fourier / Custom → 3
      - Wavelet (all subbands)     → 12
    """
    model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

    # Replace first conv if channels differ from 3
    if in_channels != 3:
        old = model.features[0][0]
        model.features[0][0] = nn.Conv2d(
            in_channels, old.out_channels,
            kernel_size=old.kernel_size, stride=old.stride,
            padding=old.padding, bias=False
        )

    # Replace classifier head: 1000 ImageNet classes → 10 CIFAR-10 classes
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    return model

print('get_model ready')

---
## Cell 5 — Training & Evaluation Utilities (shared)

In [ ]:
class History:
    """Stores train/val loss and accuracy for every epoch."""
    def __init__(self):
        self.train_loss, self.train_acc = [], []
        self.val_loss,   self.val_acc   = [], []
    def record(self, tl, ta, vl, va):
        self.train_loss.append(tl); self.train_acc.append(ta)
        self.val_loss.append(vl);   self.val_acc.append(va)


def run_epoch(model, loader, criterion, optimizer=None):
    """One training or validation pass. If optimizer is None → eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if training: loss.backward(); optimizer.step()
            preds = outputs.argmax(1)
            total_loss += loss.item() * labels.size(0)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss/total, correct/total, np.array(all_preds), np.array(all_labels)


def train_model(domain, transform_fn, in_channels):
    """Full training pipeline for one domain."""
    print(f'\n===== Training: {domain} =====')
    train_loader, val_loader, test_loader = get_dataloaders(transform_fn)

    model     = get_model(in_channels).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history   = History()

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _ = run_epoch(model, val_loader,   criterion)
        history.record(tr_loss, tr_acc, vl_loss, vl_acc)
        print(f'Epoch {epoch:02d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}')

    # Final test evaluation
    _, test_acc, test_preds, test_labels = run_epoch(model, test_loader, criterion)
    print(f'\n[{domain}] Test Accuracy: {test_acc:.4f}')

    # Save model weights
    os.makedirs('results', exist_ok=True)
    torch.save(model.state_dict(), f'results/{domain}_weights.pth')

    return model, history, test_preds, test_labels


def plot_results(domain, history, test_preds, test_labels):
    """Plots loss curves and confusion matrix for a domain."""
    epochs = range(1, len(history.train_loss) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(epochs, history.train_loss, label='Train')
    axes[0].plot(epochs, history.val_loss,   label='Val')
    axes[0].set_title(f'{domain} — Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(epochs, history.train_acc, label='Train')
    axes[1].plot(epochs, history.val_acc,   label='Val')
    axes[1].set_title(f'{domain} — Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

    cm = confusion_matrix(test_labels, test_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[2])
    axes[2].set_title(f'{domain} — Confusion Matrix')
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')

    plt.tight_layout()
    plt.show()

print('Training and evaluation utilities ready')

---
## Domain 1 — Raw Spatial (RGB)
**Subteam A: Habiba, Fareeda**

The baseline. We feed the original RGB image directly into MobileNet after resizing and normalising. No transformation is applied to the image content.

In [ ]:
# --- SPATIAL TRANSFORM ---
# Input:  (3, 32, 32) tensor in [0, 1]
# Output: (3, 224, 224) tensor — resized and normalised with ImageNet mean/std

# Steps:
#   1. Resize from 32x32 to 224x224 using F.interpolate (bilinear)
#   2. Normalise with ImageNet mean=[0.485,0.456,0.406] and std=[0.229,0.224,0.225]

def spatial_transform(image):
    pass  # replace with your code

In [ ]:
# --- VISUALISE A FEW SPATIAL IMAGES ---
# Show 6 sample images after the spatial transform
# Use plt.imshow — remember to clamp values to [0,1] and permute to (H,W,C)


In [ ]:
# --- TRAIN SPATIAL ---
spatial_model, spatial_history, spatial_preds, spatial_labels = train_model(
    domain       = 'spatial',
    transform_fn = spatial_transform,
    in_channels  = 3
)

In [ ]:
# --- RESULTS ---
plot_results('spatial', spatial_history, spatial_preds, spatial_labels)

---
## Domain 2 — Fourier Domain
**Subteam A: Habiba, Fareeda**

**What the frequency domain represents:**  
*(write here)*

**Why it may help classification:**  
*(write here)*

In [ ]:
# --- FOURIER TRANSFORM ---
# Input:  (3, 32, 32) tensor in [0, 1]
# Output: (3, 224, 224) tensor — log-magnitude FFT per channel, resized

# Steps (per channel):
#   1. Apply torch.fft.fft2() to get the 2D FFT
#   2. Apply torch.fft.fftshift() to move zero-frequency to centre
#   3. Compute magnitude: torch.abs(fft_result)
#   4. Apply log scaling: torch.log1p(magnitude)  — log(1+x) avoids log(0)
# After all 3 channels:
#   5. Stack into (3, 32, 32)
#   6. Normalise each channel to [0, 1]
#   7. Resize to 224x224

def fourier_transform(image):
    pass  # replace with your code

In [ ]:
# --- VISUALISE FFT LOG-MAGNITUDE ---
# Show 6 sample images after the fourier transform
# The images should look like bright centre patterns (low freq) fading outward


In [ ]:
# --- TRAIN FOURIER ---
fourier_model, fourier_history, fourier_preds, fourier_labels = train_model(
    domain       = 'fourier',
    transform_fn = fourier_transform,
    in_channels  = 3
)

In [ ]:
# --- RESULTS ---
plot_results('fourier', fourier_history, fourier_preds, fourier_labels)

---
## Domain 3 — Wavelet Domain
**Subteam B: Nour, Haya**

**Wavelet choice and justification:**  
*(write here)*

**Subband selection (LL only or all 4) and why:**  
*(write here)*

In [ ]:
# --- WAVELET TRANSFORM ---
# Input:  (3, 32, 32) tensor in [0, 1]
# Output: (12, 224, 224) tensor — 4 subbands x 3 channels, resized

# Steps:
#   1. Convert tensor to numpy (pywt needs numpy)
#   For each of the 3 channels:
#     2. Apply pywt.dwt2(channel, 'haar') → returns LL, (LH, HL, HH)
#        Each subband is (16, 16) — half the original size
#     3. Convert each subband back to a torch tensor
#     4. Append all 4 subbands to a list
#   After all 3 channels (12 subbands total):
#     5. Stack into (12, 16, 16)
#     6. Normalise each channel to [0, 1]
#     7. Resize to 224x224

def wavelet_transform(image):
    pass  # replace with your code

In [ ]:
# --- VISUALISE THE 4 SUBBANDS ---
# Pick one image and show its LL, LH, HL, HH subbands side by side
# LL should look like a blurry thumbnail; LH/HL/HH should show edges


In [ ]:
# --- TRAIN WAVELET ---
wavelet_model, wavelet_history, wavelet_preds, wavelet_labels = train_model(
    domain       = 'wavelet',
    transform_fn = wavelet_transform,
    in_channels  = 12   # 4 subbands x 3 channels
)

In [ ]:
# --- RESULTS ---
plot_results('wavelet', wavelet_history, wavelet_preds, wavelet_labels)

---
## Domain 4 — Custom Domain
**Subteam B: Nour, Haya**

**Chosen domain:**  
*(write here)*

**Justification — what does this domain capture and why is it useful:**  
*(write here)*

In [ ]:
# --- CUSTOM TRANSFORM ---
# Input:  (3, 32, 32) tensor in [0, 1]
# Output: (C, 224, 224) tensor
# Write your steps as comments before implementing

def custom_transform(image):
    pass  # replace with your code

In [ ]:
# --- VISUALISE CUSTOM TRANSFORMED IMAGES ---


In [ ]:
# --- TRAIN CUSTOM ---
# Change in_channels to match your transform output
custom_model, custom_history, custom_preds, custom_labels = train_model(
    domain       = 'custom',
    transform_fn = custom_transform,
    in_channels  = 3   # change if your transform outputs more channels
)

In [ ]:
# --- RESULTS ---
plot_results('custom', custom_history, custom_preds, custom_labels)

---
## Final Comparison — All 4 Domains

In [ ]:
# --- ACCURACY SUMMARY TABLE ---
# Print a table comparing best val accuracy and test accuracy across all 4 domains

results = {
    'spatial': (spatial_history, spatial_preds, spatial_labels),
    'fourier': (fourier_history, fourier_preds, fourier_labels),
    'wavelet': (wavelet_history, wavelet_preds, wavelet_labels),
    'custom':  (custom_history,  custom_preds,  custom_labels),
}

print(f'{"Domain":<10} {"Best Val Acc":>14} {"Test Acc":>10}')
print('-' * 36)
for domain, (hist, preds, labels) in results.items():
    test_acc = (preds == labels).mean()
    print(f'{domain:<10} {max(hist.val_acc):>14.4f} {test_acc:>10.4f}')

In [ ]:
# --- VALIDATION ACCURACY COMPARISON PLOT ---
# All 4 domains on the same graph

plt.figure(figsize=(10, 5))
for domain, (hist, _, _) in results.items():
    plt.plot(range(1, NUM_EPOCHS+1), hist.val_acc, label=domain)
plt.title('Validation Accuracy — All Domains')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()